## Honey-Bee Hive Health Prediction: Experimental modeling

**Info:** In this notebook, we will implement a series of models on our curated cross-validation dataset containing Honey-Bee health descriptors and local weather features to predict weather a hive in a given apiary is healthy in the upcoming inspection. The goal is to beat the performance of baseline model by increasing the complexity of our model.

**Target variable**: Health status (Healthy)

**Descriptive Features:** Past Health Descriptors and Weather trends

**Train-Test split:** GroupShuffleSplit

**Cross-validation split:** GroupKFold (Group key: HiveID)

**Models:** Logistic Regression, Random Forests, XGBoost and LightGBM

In [ ]:
# Imports from standard libraries
import pandas as pd
import numpy as np
import pickle
import warnings

# Imports from sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate, GroupKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, roc_auc_score, make_scorer, accuracy_score
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import MinMaxScaler

# Imports from other libraries
import lightgbm as lgb
import xgboost as xgb

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='Found whitespace in feature_names')

#### Testing various models on the training set and testing on the hold-out set

In [ ]:
try:
    # 1. Load Data
    print("Loading training set...")
    path_to_data = '../../data/test_train/training_data.pkl'
    with open(path_to_data, 'rb') as f:
        loaded_data = pickle.load(f)

    X_train = loaded_data["X_train"]
    groups_train = loaded_data["groups_train"]
    Y_train = loaded_data["Y_train"]

    # 2. Define Target and Features
    target = 'Healthy'
    group_key = 'HiveID'

    # 2.1 Historical and Weather Features
    health_history = [
        'Is_First_Inspection', 'Days_Since_Last_Inspection', 'Hive_Age_Days',
        'Prev_Brood_Status', 'Prev_Bees_Status', 'Prev_Queen_Status',
        'Prev_Food_Status', 'Prev_Stressors_Status', 'Prev_Space_Status']
        # Removed 'Prev_Health_Status'
    weather_features = [
        'Avg_prcp', 'Avg_wind', 'Avg_tmax', 'Avg_tmin', 'Avg_tavg',
        'Avg_snow', 'Num_frost_days']

    health_weather_features = health_history + weather_features
    features_to_use = health_weather_features

    # 3. Print Data Shapes
    print("\n--- DATA SHAPEs ---")
    print(f"X shape: {X_train.shape}")
    print(f"Y shape: {Y_train.shape}")

    # 4. Calculate scale_pos_weight for imbalance
    scale_pos_weight = (Y_train == 0).sum() / (Y_train == 1).sum()
    print(f"\nCalculated scale_pos_weight for imbalance: {scale_pos_weight:.2f}")

    # 5. Create Preprocessing Pipeline
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value=-1)),
        ('scaler', MinMaxScaler()) 
    ])
    preprocessor = ColumnTransformer(
        transformers=[('num', numeric_transformer, features_to_use)],
        remainder='passthrough'
    )

    # 6. Define All Models
    models_to_run = {
        'LogisticRegression': LogisticRegression(
            random_state=42, max_iter=2000, class_weight='balanced', n_jobs=-1
        ),
        'RandomForest': RandomForestClassifier(
            random_state=42, n_estimators=300, max_depth=8,
            class_weight='balanced', n_jobs=-1
        ),
        'XGBoost': xgb.XGBClassifier( 
            random_state=42, n_estimators=300, learning_rate=0.05,
            max_depth=5, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight, n_jobs=-1
        ),
        'LightGBM': lgb.LGBMClassifier(
            random_state=42, n_estimators=300, learning_rate=0.05,
            max_depth=5, subsample=0.8, colsample_bytree=0.8,
            class_weight='balanced', n_jobs=-1, verbose=-1
        )
    }

    # 6. Define Cross-Validation Strategy
    gkf_cv = GroupKFold(n_splits=5)

    # 7. Define Scorers
    scorers = {
        'f1_score': make_scorer(f1_score, pos_label=1),
        'roc_auc': make_scorer(roc_auc_score),
        'accuracy': make_scorer(accuracy_score)
    }

    # 8. Run Cross-Validation on All Models
    print(f"\n--- Running 5-fold GroupKFold CV ---")
    
    cv_results_all = {}
    for name, model in models_to_run.items():
        print(f"Running CV for: {name}...")
        
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', model)
        ])
        
        cv_results = cross_validate(
            pipeline, 
            X_train, 
            Y_train,
            groups=groups_train,
            cv=gkf_cv,
            scoring=scorers, 
            n_jobs=-1, 
            verbose=0
        )
        cv_results_all[name] = cv_results
        
        print(f"Average F1 Score:   {np.mean(cv_results['test_f1_score']):.3f} +/- {np.std(cv_results['test_f1_score']):.3f}")
        print(f"Average ROC-AUC Score: {np.mean(cv_results['test_roc_auc']):.3f} +/- {np.std(cv_results['test_roc_auc']):.3f}")
        print(f"Average Accuracy: {np.mean(cv_results['test_accuracy']):.3f} +/- {np.std(cv_results['test_accuracy']):.3f}")

except FileNotFoundError:
    print("\n--- ERROR ---")
    print(f"Could not find the file '{path_to_data}'.")
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()

Loading training set...

--- DATA SHAPEs ---
X shape: (1596, 16)
Y shape: (1596,)

Calculated scale_pos_weight for imbalance: 1.69

--- Running 5-fold GroupKFold CV ---
Running CV for: LogisticRegression...
Average F1 Score:   0.633 +/- 0.084
Average ROC-AUC Score: 0.703 +/- 0.036
Average Accuracy: 0.705 +/- 0.028
Running CV for: RandomForest...
Average F1 Score:   0.658 +/- 0.072
Average ROC-AUC Score: 0.727 +/- 0.026
Average Accuracy: 0.745 +/- 0.014
Running CV for: XGBoost...
Average F1 Score:   0.658 +/- 0.102
Average ROC-AUC Score: 0.733 +/- 0.048
Average Accuracy: 0.759 +/- 0.015
Running CV for: LightGBM...


/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Average F1 Score:   0.646 +/- 0.110
Average ROC-AUC Score: 0.722 +/- 0.052
Average Accuracy: 0.751 +/- 0.018


/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


#### Tuning Hyperparameters for XGboost (Best performer in the initial experiment)

From the above KPIs, it is clear that all the tree-based models outperform simple logistic regression, which indicates that our dataset contains some non-linear correlations. XGBoost performs slightly better compared to the other tree-based models for the choice of parameters above. Next, we will only use XGBoost to tune the hyperparameters.

In [ ]:
try:
    # 1. Load Data
    print("Loading training set...")
    path_to_data = '../../data/test_train/training_data.pkl'
    with open(path_to_data, 'rb') as f:
        loaded_data = pickle.load(f)

    X_train = loaded_data["X_train"]
    groups_train = loaded_data["groups_train"]
    Y_train = loaded_data["Y_train"]

    # 2. Define Target and Features
    target = 'Healthy'
    group_key = 'HiveID'

    # 2.1 Historical and Weather Features
    health_history = [
        'Is_First_Inspection', 'Days_Since_Last_Inspection', 'Hive_Age_Days',
        'Prev_Brood_Status', 'Prev_Bees_Status', 'Prev_Queen_Status',
        'Prev_Food_Status', 'Prev_Stressors_Status', 'Prev_Space_Status']
        # Removed 'Prev_Health_Status'
    weather_features = [
        'Avg_prcp', 'Avg_wind', 'Avg_tmax', 'Avg_tmin', 'Avg_tavg',
        'Avg_snow', 'Num_frost_days']

    health_weather_features = health_history + weather_features
    features_to_use = health_weather_features

    # 3. Print Data Shapes
    print("\n--- DATA SHAPEs ---")
    print(f"X shape: {X_train.shape}")
    print(f"Y shape: {Y_train.shape}")

    # 4. Calculate scale_pos_weight for imbalance
    scale_pos_weight = (Y_train == 0).sum() / (Y_train == 1).sum()
    print(f"Calculated scale_pos_weight for imbalance: {scale_pos_weight:.2f}")

    # 5. Create Preprocessing Pipeline
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value=-1)),
        ('scaler', MinMaxScaler()) 
    ])
    preprocessor = ColumnTransformer(
        transformers=[('num', numeric_transformer, features_to_use)],
        remainder='passthrough'
    )

    # 6. Define Model (XGBoost for Hyperparameter Tuning)
    model = xgb.XGBClassifier(
            random_state=42, n_estimators=300, learning_rate=0.05,
            max_depth=5, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight, n_jobs=-1
        )
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # 7. Define Search Space and CV Splitter for Hyperparameter Tuning
    param_dist = {
        'model__n_estimators': [100, 200, 300, 500],
        'model__max_depth': [3, 5, 7, 10],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__subsample': [0.7, 0.9, 1.0],
        'model__colsample_bytree': [0.7, 0.9, 1.0]
    }
    
    # 8. Define Cross-Validation Strategy
    gkf_for_tuning = GroupKFold(n_splits=5)

    # 9. Set up and Run RandomizedSearchCV on TRAINING data only 
    random_search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_dist,
        n_iter=50,
        cv=gkf_for_tuning,
        scoring=make_scorer(roc_auc_score),
        n_jobs=-1,
        verbose=1,
        random_state=42
    )

    print("\nRunning Randomized Hyperparameter Search (on X_train, grouped by HiveID)...")
    random_search.fit(X_train, Y_train, groups=groups_train)

    print("\n--- TUNING COMPLETE ---")
    print(f"Best CV ROC-AUC Score: {random_search.best_score_:.4f}")
    print(f"Best Parameters: {random_search.best_params_}")

except FileNotFoundError:
    print("\n--- ERROR ---")
    print(f"Could not find the file '{path_to_data}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Loading training set...

--- DATA SHAPEs ---
X shape: (1596, 16)
Y shape: (1596,)
Calculated scale_pos_weight for imbalance: 1.69

Running Randomized Hyperparameter Search (on X_train, grouped by HiveID)...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

--- TUNING COMPLETE ---
Best CV ROC-AUC Score: 0.7478
Best Parameters: {'model__subsample': 0.7, 'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.9}
